# 02 — Advanced ESC-50: P1 Ensemble & P2 CNN–Transformer + EMA + TTA

**P1:** Two-view, dual-architecture ensemble: COCO-pretrained ResNet-tiny versus ImageNet-pretrained EfficientNet-B0, with different log-mel resolutions. Two independently trained classifiers, fixed equal probability voting (no test-fold tuning).

**P2:** ResNet-tiny pretrained CNN + lightweight Transformer encoder; exponential moving average (EMA) weights and deterministic multi-view time-shift TTA.

**Bukan implementasi identik** penelitian WASF atau PTFMSAN; ini adaptasi rancangan sendiri yang terinspirasi gagasan fusion, multi-scale dan modeling temporal. Tidak memakai knowledge distillation.

Literatur: [WASF (2026), DOI 10.1016/j.asoc.2025.114192](https://doi.org/10.1016/j.asoc.2025.114192); [PTFMSAN (2025), DOI 10.3390/e27101007](https://doi.org/10.3390/e27101007).

**Protokol:** ESC-50 official 5 folds; 4 train, 1 test; 500 epochs; Adam 1e-4; weight decay 1e-3; MultiStepLR; baseline masking, roll and mixup. Final epoch (not best test epoch). No independent validation. Report inference cost and parameters for edge-device tradeoff.

**PENTING untuk Kaggle:** semua hasil JSON/CSV/checkpoint ditulis ke `/kaggle/working/outputs_advanced/` (bukan `/kaggle/input`). `Save Version → Save & Run All` menyimpan output versi yang selesai. `Quick Save` menyimpan notebook dan tidak menjamin hasil sesi interaktif menjadi output versi baru. Di notebook 03, attach **Notebook Output** versi tersimpan dari 02 dan baseline.

In [1]:
import os
import sys
import json
import math
import time
import copy
import random
import hashlib
import subprocess
import contextlib
import platform
import urllib.request
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchaudio
import soundfile as sf
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, f1_score

# ------------------------------------------------------------------
# PROJECT DIRECTORY: semua output disimpan di dalam folder ini
# ------------------------------------------------------------------
def find_project_dir():
    env = os.environ.get("PROJECT_DIR")                 # override manual
    if env:
        return Path(env).expanduser().resolve()
    if Path("/kaggle/working").exists():                # Kaggle: hanya /kaggle/working yang ikut tersimpan
        return Path("/kaggle/working")
    return Path.cwd().resolve()                         # lokal / Jupyter: folder kerja notebook

PROJECT_DIR = find_project_dir()
WORK        = (Path("/kaggle/temp") / "esc50_repro") if Path("/kaggle/temp").exists() else PROJECT_DIR / "esc50_repro"               # repos, weights, dataset, cache (besar, bisa diunduh ulang)
OUT_DIR     = PROJECT_DIR / "outputs"                   # HASIL yang harus tersimpan
RESULTS_DIR = OUT_DIR / "results"
SMOKE_DIR   = OUT_DIR / "results_smoke"
CKPT_DIR    = OUT_DIR / "checkpoints"
FIG_DIR     = OUT_DIR / "figures"
LOG_DIR     = OUT_DIR / "logs"
for _d in (WORK, RESULTS_DIR, SMOKE_DIR, CKPT_DIR, FIG_DIR, LOG_DIR):
    _d.mkdir(parents=True, exist_ok=True)


# ---- helper penyimpanan: tulis ke file sementara -> fsync -> rename atomik -> verifikasi ----
def durable_write(path, data, mode="w"):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    with open(tmp, mode) as f:
        f.write(data); f.flush(); os.fsync(f.fileno())
    os.replace(tmp, path)
    assert path.exists() and path.stat().st_size > 0, f"GAGAL menyimpan: {path}"
    return path

def durable_torch_save(obj, path):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    with open(tmp, "wb") as f:
        torch.save(obj, f); f.flush(); os.fsync(f.fileno())
    os.replace(tmp, path)
    assert path.exists() and path.stat().st_size > 0, f"GAGAL menyimpan: {path}"
    return path

def save_json(path, obj):
    return durable_write(path, json.dumps(obj, indent=2, default=str))

def log(msg):
    """print + tambahkan ke outputs/logs/run_log.txt"""
    line = f"[{time.strftime('%Y-%m-%d %H:%M:%S')}] {msg}"
    print(line)
    with open(LOG_DIR / "run_log.txt", "a") as f:
        f.write(line + "\n"); f.flush()

durable_write(OUT_DIR / ".write_test", "ok")           # gagal keras jika folder tidak bisa ditulis
print("PROJECT_DIR :", PROJECT_DIR)
print("OUT_DIR     :", OUT_DIR)

C = SimpleNamespace(
    # ---- audio / features (paper Sect. 3.1 and 4) ----
    sr=22050, dur=10.0, n_mels=128,
    win_hops=[(551, 220), (1102, 551), (2205, 1102)],
    width=250,
    # ---- NOT given in the paper: assumptions ----
    n_fft=None,            # None -> next power of two >= window length
    log_eps=1e-5,
    norm_mode="passt",     # "passt": (x+4.5)/5   |  "paper": (x-4.5)/5
    pad_position="end",    # ESC-50 clips are 5 s -> padded to 10 s ("end" or "center")
    freq_mask=24, time_mask=40,   # max mask widths (bins / frames)
    mixup_prob=1.0,
    amp=True,              # fp16 autocast (paper did not state)
    seed=42,
    # ---- training (paper Sect. 8) ----
    epochs=500, batch_size=64, lr=1e-4, weight_decay=1e-3,
    lr_milestones=(1/3, 2/3), lr_gamma=0.1,
    mixup_alpha=0.2,
    # ---- what to run ----
    models=["resnet_tiny"],
    folds=[1, 2, 3, 4, 5],
    eval_every=50,         # monitoring only; reported number is the LAST epoch
    ckpt_every=10,         # simpan checkpoint resume tiap N epoch (bukan tiap epoch -> jauh lebih cepat)
)
print(C)
OUT_DIR=PROJECT_DIR/'outputs_advanced'; RESULTS_DIR=OUT_DIR/'results'; CKPT_DIR=OUT_DIR/'checkpoints'; LOG_DIR=OUT_DIR/'logs'
for d in [OUT_DIR,RESULTS_DIR,CKPT_DIR,LOG_DIR]:d.mkdir(parents=True,exist_ok=True)
RUN_METHODS=['P1_ensemble','P2_hybrid_ema_tta']; RUN_FOLDS=[1,2,3,4,5]
SAVE_WEIGHTS=False # True: simpan weights setiap fold (besar); JSON selalu disimpan.
C.ckpt_every=50 # resume checkpoint sementara setiap 50 epoch


/root/acdnet/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PROJECT_DIR : /root/acdnet
OUT_DIR     : /root/acdnet/outputs
namespace(sr=22050, dur=10.0, n_mels=128, win_hops=[(551, 220), (1102, 551), (2205, 1102)], width=250, n_fft=None, log_eps=1e-05, norm_mode='passt', pad_position='end', freq_mask=24, time_mask=40, mixup_prob=1.0, amp=True, seed=42, epochs=500, batch_size=64, lr=0.0001, weight_decay=0.001, lr_milestones=(0.3333333333333333, 0.6666666666666666), lr_gamma=0.1, mixup_alpha=0.2, models=['resnet_tiny'], folds=[1, 2, 3, 4, 5], eval_every=50, ckpt_every=10)


## 1. Environment dan backbone COCO-pretrained

In [2]:
def sh(cmd):
    print("$", cmd)
    subprocess.run(cmd, shell=True, check=True)

try:
    import torch, torchvision, torchaudio
except ImportError as e:
    raise SystemExit("Install PyTorch + torchvision + torchaudio (CUDA build) first: https://pytorch.org/get-started/locally/") from e

sh(f"{sys.executable} -m pip install -q opencv-python-headless pyyaml scipy matplotlib pandas seaborn tqdm requests thop pillow soundfile")

YOLOV7_DIR = WORK / "yolov7"
SCALED_DIR = WORK / "ScaledYOLOv4"
ESC_DIR    = WORK / "ESC-50"
if not YOLOV7_DIR.exists(): sh(f'git clone --depth 1 https://github.com/WongKinYiu/yolov7.git "{YOLOV7_DIR}"')
if not SCALED_DIR.exists(): sh(f'git clone --depth 1 --branch yolov4-csp https://github.com/WongKinYiu/ScaledYOLOv4.git "{SCALED_DIR}"')
if not ESC_DIR.exists():    sh(f'git clone --depth 1 https://github.com/karolpiczak/ESC-50.git "{ESC_DIR}"')   # ~600 MB

import urllib.request
W_TINY = WORK / "yolov7-tiny.pt"           # COCO weights, YOLOv7-tiny
W_V4   = WORK / "yolov4-csp.weights"       # COCO weights, Scaled-YOLOv4 CSP (PyTorch-repo version)
for url, dst in [("https://github.com/WongKinYiu/yolov7/releases/download/v0.1/yolov7-tiny.pt", W_TINY),
                 ("https://github.com/WongKinYiu/ScaledYOLOv4/releases/download/weights/yolov4-csp.weights", W_V4)]:
    if not dst.exists():
        print("downloading", dst.name); urllib.request.urlretrieve(url, dst)

import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
import torchaudio, soundfile as sf
from tqdm.auto import tqdm

assert torch.cuda.is_available(), "No CUDA GPU visible."
dev = torch.device("cuda")
torch.backends.cudnn.benchmark = True
print("torch", torch.__version__, "|", torch.cuda.get_device_name(0))

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

$ /root/acdnet/.venv/bin/python -m pip install -q opencv-python-headless pyyaml scipy matplotlib pandas seaborn tqdm requests thop pillow soundfile
torch 2.15.0+cu132 | NVIDIA RTX A4000


In [3]:
@contextlib.contextmanager
def repo_on_path(repo_dir):
    def purge():
        for k in list(sys.modules):
            if k in ("models", "utils") or k.startswith(("models.", "utils.")):
                del sys.modules[k]
    purge(); sys.path.insert(0, str(repo_dir))
    try:
        yield
    finally:
        sys.path.remove(str(repo_dir)); purge()


class YoloV7Backbone(nn.Module):
    """Runs yolov7 layers sequentially, honouring each layer's `.f` (same logic as Model.forward_once)."""
    def __init__(self, layers):
        super().__init__()
        self.layers = nn.ModuleList(layers)
    def forward(self, x):
        y = []
        for m in self.layers:
            if m.f != -1:
                x = y[m.f] if isinstance(m.f, int) else [x if j == -1 else y[j] for j in m.f]
            x = m(x)
            y.append(x)
        return x


class DarknetBackbone(nn.Module):
    """Runs the first N modules of a ScaledYOLOv4 Darknet (convolutional / route / shortcut only)."""
    def __init__(self, modules):
        super().__init__()
        self.module_list = nn.ModuleList(modules)
    def forward(self, x):
        out = []
        for m in self.module_list:
            if m.__class__.__name__ in ("WeightedFeatureFusion", "FeatureConcat", "FeatureConcat2", "FeatureConcat3", "FeatureConcat_l"):
                x = m(x, out)
            else:
                x = m(x)
            out.append(x)
        return x


def build_yolov7_tiny_backbone():
    with repo_on_path(YOLOV7_DIR):
        try:
            ckpt = torch.load(W_TINY, map_location="cpu", weights_only=False)
        except TypeError:                      # old torch without weights_only
            ckpt = torch.load(W_TINY, map_location="cpu")
        full = ckpt.get("ema")
        full = full if full is not None else ckpt["model"]
        full = full.float()
    layers = list(full.model.children())[:29]
    assert hasattr(layers[0], "bn"), "Checkpoint looks fused (no BN); need the unfused training checkpoint."
    assert layers[28].conv.out_channels == 512
    bb = YoloV7Backbone(layers)
    for p in bb.parameters(): p.requires_grad_(True)
    return bb.train()


def build_yolov4csp_backbone():
    with repo_on_path(SCALED_DIR):
        from models.models import Darknet, load_darknet_weights
        net = Darknet(str(SCALED_DIR / "models" / "yolov4-csp.cfg"), (128, 250))
        load_darknet_weights(net, str(W_V4), cutoff=99)       # only layers 0..98 (backbone)
    mods = list(net.module_list)[:99]
    assert mods[98][0].out_channels == 1024
    bb = DarknetBackbone(mods)
    for p in bb.parameters(): p.requires_grad_(True)
    return bb.train()


class AudioNet(nn.Module):
    def __init__(self, backbone, c_out, n_classes):
        super().__init__()
        self.backbone = backbone
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(c_out, 1024), nn.Hardswish(), nn.Dropout(0.2),
            nn.Linear(1024, n_classes))
    def forward(self, x):
        return self.head(self.backbone(x))


SPECS = {
    "resnet_tiny": dict(builder=build_yolov7_tiny_backbone, c_out=512,  paper_params=2_996_903,  paper_acc=81.6),
    "resnet":      dict(builder=build_yolov4csp_backbone,   c_out=1024, paper_params=26_610_535, paper_acc=81.9),
}
_PRETRAINED = {}
def build_model(name, n_classes):
    if name not in _PRETRAINED:
        _PRETRAINED[name] = SPECS[name]["builder"]()
    return AudioNet(copy.deepcopy(_PRETRAINED[name]), SPECS[name]["c_out"], n_classes)

def n_params(m): return sum(p.numel() for p in m.parameters())

## 2. Fitur audio dua resolusi dan cache

In [4]:
class LogMelFront(nn.Module):
    def __init__(self, C):
        super().__init__()
        self.C = C
        self.mels = nn.ModuleList()
        for win, hop in C.win_hops:
            n_fft = C.n_fft or (1 << (win - 1).bit_length())
            self.mels.append(torchaudio.transforms.MelSpectrogram(
                sample_rate=C.sr, n_fft=n_fft, win_length=win, hop_length=hop,
                f_min=0.0, f_max=C.sr / 2, n_mels=C.n_mels, power=2.0))
        self.mean = {"passt": -4.5, "paper": 4.5}[C.norm_mode]
        self.std = 5.0
    @torch.no_grad()
    def forward(self, wav):                               # wav: (B, samples)
        chans = []
        for m in self.mels:
            s = torch.log(m(wav) + self.C.log_eps)        # (B, n_mels, T)
            s = F.interpolate(s.unsqueeze(1), size=(self.C.n_mels, self.C.width),
                              mode="bilinear", align_corners=False)
            chans.append(s)
        x = torch.cat(chans, 1)                           # (B, 3, 128, 250)
        return (x - self.mean) / self.std


def load_wave(path, C):
    x, sr = sf.read(str(path), dtype="float32", always_2d=False)
    if x.ndim > 1: x = x.mean(1)
    x = torch.from_numpy(x)
    if sr != C.sr: x = torchaudio.functional.resample(x, sr, C.sr)
    n = int(round(C.sr * C.dur))
    if x.numel() < n:
        pad = n - x.numel()
        left = 0 if C.pad_position == "end" else pad // 2
        x = F.pad(x, (left, pad - left))
    elif x.numel() > n:
        s = (x.numel() - n) // 2
        x = x[s:s + n]
    return x


def build_dataset(C):
    key = json.dumps([C.sr, C.dur, C.n_mels, C.win_hops, C.width, C.n_fft, C.log_eps, C.norm_mode, C.pad_position])
    cache = WORK / f"esc50_spec_{hashlib.md5(key.encode()).hexdigest()[:8]}.pt"
    if cache.exists():
        print("loading cache", cache.name); return torch.load(cache)
    meta = pd.read_csv(ESC_DIR / "meta" / "esc50.csv")
    front = LogMelFront(C).to(dev)
    X = torch.empty(len(meta), 3, C.n_mels, C.width, dtype=torch.float16)
    for i in tqdm(range(0, len(meta), 50), desc="spectrograms"):
        rows = meta.iloc[i:i + 50]
        wav = torch.stack([load_wave(ESC_DIR / "audio" / f, C) for f in rows.filename]).to(dev)
        X[i:i + len(rows)] = front(wav).half().cpu()
    data = dict(X=X, y=torch.tensor(meta.target.values, dtype=torch.long),
                fold=torch.tensor(meta.fold.values, dtype=torch.long))
    torch.save(data, cache)
    return data



In [5]:
# View B: alternate window/hop configurations, same normalized 3×128×250 input.
# This is NOT MFCC/STFT fusion; both views remain log-mel to retain pretrained RGB-like input compatibility.
C_ALT=SimpleNamespace(**vars(C))
C_ALT.win_hops=[(330,165),(1654,551),(3308,1102)]
print('View A:',C.win_hops,'View B:',C_ALT.win_hops)


View A: [(551, 220), (1102, 551), (2205, 1102)] View B: [(330, 165), (1654, 551), (3308, 1102)]


## 3. Training, EMA, Transformer, TTA and durable per-fold outputs

In [6]:
from sklearn.metrics import confusion_matrix
import shutil

# Reuse baseline augment, AMP scaler, checkpoint utilities.
def augment(x, C):
    B, _, Fq, T = x.shape
    d = x.device
    # random circular shift along time ("rolling")
    shifts = torch.randint(0, T, (B,), device=d)
    idx = (torch.arange(T, device=d)[None, :] - shifts[:, None]) % T
    x = torch.gather(x, 3, idx[:, None, None, :].expand(-1, x.shape[1], Fq, -1))
    # frequency masking
    if C.freq_mask > 0:
        w = torch.randint(0, C.freq_mask + 1, (B, 1), device=d)
        f0 = (torch.rand(B, 1, device=d) * (Fq - w + 1).float()).long()
        ar = torch.arange(Fq, device=d)[None, :]
        x = x.masked_fill(((ar >= f0) & (ar < f0 + w))[:, None, :, None], 0.0)
    # time masking
    if C.time_mask > 0:
        w = torch.randint(0, C.time_mask + 1, (B, 1), device=d)
        t0 = (torch.rand(B, 1, device=d) * (T - w + 1).float()).long()
        ar = torch.arange(T, device=d)[None, :]
        x = x.masked_fill(((ar >= t0) & (ar < t0 + w))[:, None, None, :], 0.0)
    return x


def make_scaler(enabled):
    try:    return torch.amp.GradScaler("cuda", enabled=enabled)
    except Exception: return torch.cuda.amp.GradScaler(enabled=enabled)


def train_step(model, opt, scaler, xb, yb, C):
    xb = augment(xb, C)
    lam, perm = 1.0, None
    if C.mixup_alpha > 0 and random.random() < C.mixup_prob:
        lam = float(np.random.beta(C.mixup_alpha, C.mixup_alpha))
        perm = torch.randperm(xb.size(0), device=xb.device)
        xb = lam * xb + (1 - lam) * xb[perm]
    with torch.autocast("cuda", dtype=torch.float16, enabled=C.amp):
        out = model(xb)
        loss = F.cross_entropy(out.float(), yb) if perm is None else \
               lam * F.cross_entropy(out.float(), yb) + (1 - lam) * F.cross_entropy(out.float(), yb[perm])
    opt.zero_grad(set_to_none=True)
    scaler.scale(loss).backward()
    scaler.step(opt); scaler.update()
    return loss.item()

@torch.no_grad()
def evaluate(model, X, y, C, bs=100):
    model.eval()

    all_preds = []
    all_targets = []

    for i in range(0, len(X), bs):

        with torch.autocast(
            "cuda",
            dtype=torch.float16,
            enabled=C.amp
        ):
            out = model(X[i:i + bs].float())

        preds = out.argmax(1)

        all_preds.extend(preds.cpu().numpy())
        all_targets.extend(y[i:i + bs].cpu().numpy())

    accuracy = accuracy_score(all_targets, all_preds)
    macro_f1 = f1_score(
        all_targets,
        all_preds,
        average="macro",
        zero_division=0
    )

    weighted_f1 = f1_score(
        all_targets,
        all_preds,
        average="weighted",
        zero_division=0
    )

    model.train()

    return {
        "accuracy": accuracy * 100,
        "macro_f1": macro_f1 * 100,
        "weighted_f1": weighted_f1 * 100
    }


def save_checkpoint(path, epoch, model, opt, sched, scaler, hist):
    checkpoint = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": opt.state_dict(),
        "scheduler_state_dict": sched.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "history": hist,
        "python_rng": random.getstate(),
        "numpy_rng": np.random.get_state(),
        "torch_rng": torch.get_rng_state(),
        "cuda_rng": torch.cuda.get_rng_state_all()
    }
    durable_torch_save(checkpoint, path)


def load_checkpoint(path, model, opt, sched, scaler):
    checkpoint = torch.load(
        path,
        map_location=dev,
        weights_only=False
    )
    model.load_state_dict(checkpoint["model_state_dict"])
    opt.load_state_dict(checkpoint["optimizer_state_dict"])
    sched.load_state_dict(checkpoint["scheduler_state_dict"])
    scaler.load_state_dict(checkpoint["scaler_state_dict"])
    random.setstate(checkpoint["python_rng"])
    np.random.set_state(checkpoint["numpy_rng"])
    torch.set_rng_state(checkpoint["torch_rng"].cpu())
    torch.cuda.set_rng_state_all([s.cpu() for s in checkpoint["cuda_rng"]])
    return checkpoint["epoch"], checkpoint["history"]




In [9]:
def build_efficientnet_b0(n_classes):
    # ImageNet-pretrained EfficientNet-B0. Requires internet on first run.
    from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
    model=efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
    in_features=model.classifier[-1].in_features
    model.classifier[-1]=nn.Linear(in_features,n_classes)
    return model

class HybridCNNTransformer(nn.Module):
    """Pretrained YOLOv7-tiny CNN + compact temporal Transformer.
    Tokenization averages the CNN frequency axis, retaining temporal sequence.
    """
    def __init__(self, base, d_model=128, n_heads=4):
        super().__init__()
        self.backbone=base.backbone
        self.project=nn.Linear(512,d_model)
        layer=nn.TransformerEncoderLayer(d_model=d_model,nhead=n_heads,dim_feedforward=256,
                                          dropout=0.1,batch_first=True,norm_first=True)
        self.encoder=nn.TransformerEncoder(layer,num_layers=1,enable_nested_tensor=False)
        self.norm=nn.LayerNorm(d_model)
        self.head=nn.Sequential(nn.Linear(d_model,128),nn.Hardswish(),nn.Dropout(0.2),nn.Linear(128,50))
    def forward(self,x):
        feat=self.backbone(x)              # (B,512,F,T)
        tokens=feat.mean(dim=2).transpose(1,2)  # (B,T,512)
        tokens=self.project(tokens)
        # Sinusoidal temporal position encoding, computed on device and dtype.
        T=tokens.shape[1]; D=tokens.shape[2]
        pos=torch.arange(T,device=tokens.device,dtype=torch.float32)[:,None]
        freq=torch.exp(torch.arange(0,D,2,device=tokens.device,dtype=torch.float32)*(-math.log(10000.0)/D))
        pe=torch.zeros(T,D,device=tokens.device,dtype=tokens.dtype)
        pe[:,0::2]=torch.sin(pos*freq).to(tokens.dtype)
        pe[:,1::2]=torch.cos(pos*freq).to(tokens.dtype)
        z=self.encoder(tokens+pe[None])
        return self.head(self.norm(z.mean(dim=1)))

@torch.no_grad()
def ema_update(ema, model, decay=0.995):
    for p_ema,p in zip(ema.parameters(),model.parameters()):
        p_ema.lerp_(p.detach(),1-decay)
    for b_ema,b in zip(ema.buffers(),model.buffers()):
        b_ema.copy_(b)   # keep BN buffers synchronized

def train_batch(models,optimizers,scalers,xa,xb,y,C):
    # Same augmentation parameters as baseline; independently sampled for each branch.
    # Mixup indices and lambda shared across both branches for consistent target pairs.
    lam=1.0; perm=None
    if C.mixup_alpha>0 and random.random()<C.mixup_prob:
        lam=float(np.random.beta(C.mixup_alpha,C.mixup_alpha))
        perm=torch.randperm(len(y),device=y.device)
    losses=[]
    for model,opt,scaler,x in zip(models,optimizers,scalers,[xa,xb]):
        x=augment(x,C)
        if perm is not None: x=lam*x+(1-lam)*x[perm]
        with torch.autocast('cuda',dtype=torch.float16,enabled=C.amp):
            logits=model(x)
            loss=F.cross_entropy(logits.float(),y) if perm is None else (lam*F.cross_entropy(logits.float(),y)+(1-lam)*F.cross_entropy(logits.float(),y[perm]))
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.step(opt);scaler.update()
        losses.append(float(loss.detach().item()))
    return float(np.mean(losses))

@torch.no_grad()
def predict_probs(models, xa, xb, method, C, bs=64):
    for m in models:m.eval()
    probs=[]
    for i in range(0,len(xa),bs):
        a=xa[i:i+bs].float()
        b=xb[i:i+bs].float()
        with torch.autocast('cuda',dtype=torch.float16,enabled=C.amp):
            if method=='P1_ensemble':
                p=(models[0](a).float().softmax(1)+models[1](b).float().softmax(1))/2
            else:
                # deterministic TTA. No test-label-based choice of shifts.
                p=torch.stack([models[0](torch.roll(a,shifts=s,dims=-1)).float().softmax(1)
                               for s in (0,a.shape[-1]//4,a.shape[-1]//2)]).mean(0)
        probs.append(p.cpu())
    return torch.cat(probs,dim=0)

def run_advanced_fold(method,fold,data_a,data_b=None, SAVE_WEIGHTS=True):
    out=RESULTS_DIR/f'{method}_fold{fold}.json'
    if out.exists():
        r=json.loads(out.read_text())
        if r.get('epoch')==C.epochs:
            print('Already completed:',out);return r
        raise RuntimeError(f'Unexpected incomplete result: {out}')
    set_seed(C.seed+fold)
    tr=data_a['fold']!=fold;te=~tr
    ya=data_a['y']; fa=data_a['fold']
    assert len(ya)==2000 and int(te.sum())==400
    if method=='P1_ensemble':
        assert data_b is not None
        assert torch.equal(ya,data_b['y']) and torch.equal(fa,data_b['fold'])
    xa=data_a['X'][tr].to(dev).float(); xe=data_a['X'][te].to(dev).float()
    xb=(data_b['X'][tr].to(dev).float() if method=='P1_ensemble' else xa)
    be=(data_b['X'][te].to(dev).float() if method=='P1_ensemble' else xe)
    yt=ya[tr].to(dev); ytest=ya[te].cpu().numpy()
    if method=='P1_ensemble':
        models=[build_model('resnet_tiny',50).to(dev),build_efficientnet_b0(50).to(dev)]
    else:
        models=[HybridCNNTransformer(build_model('resnet_tiny',50)).to(dev)]
    opts=[torch.optim.Adam(m.parameters(),lr=C.lr,weight_decay=C.weight_decay) for m in models]
    sch=[torch.optim.lr_scheduler.MultiStepLR(o,[round(C.epochs/3),round(2*C.epochs/3)],gamma=C.lr_gamma) for o in opts]
    scalers=[make_scaler(C.amp) for _ in models]
    ema=copy.deepcopy(models[0]).to(dev).eval() if method=='P2_hybrid_ema_tta' else None
    if ema is not None:
        for p in ema.parameters():p.requires_grad_(False)
    hist=[]; ckpt=CKPT_DIR/f'{method}_fold{fold}_resume.pt'; start=1
    if ckpt.exists():
        z=torch.load(ckpt,map_location='cpu',weights_only=False)
        for m,s in zip(models,z['models']):m.load_state_dict(s)
        for o,s in zip(opts,z['opts']):o.load_state_dict(s)
        for s0,s in zip(sch,z['sched']):s0.load_state_dict(s)
        for s0,s in zip(scalers,z['scalers']):s0.load_state_dict(s)
        if ema is not None:ema.load_state_dict(z['ema'])
        hist=z['history'];start=z['epoch']+1
        random.setstate(z['python_rng']);np.random.set_state(z['numpy_rng'])
        torch.set_rng_state(z['torch_rng']);torch.cuda.set_rng_state_all(z['cuda_rng'])
        print('Resuming',method,fold,'epoch',start)
    start_time=time.time()
    for ep in tqdm(range(start,C.epochs+1),desc=f'{method} fold{fold}'):
        for m in models:m.train()
        idx=torch.randperm(len(yt),device=dev); losses=[]
        for k in range(0,len(idx),C.batch_size):
            ids=idx[k:k+C.batch_size]
            if len(ids)<2:continue
            if method=='P1_ensemble':
                losses.append(train_batch(models,opts,scalers,xa[ids],xb[ids],yt[ids],C))
            else:
                losses.append(train_step(models[0],opts[0],scalers[0],xa[ids],yt[ids],C))
                ema_update(ema,models[0],decay=0.995)
        for s in sch:s.step()
        rec={'epoch':ep,'loss':float(np.mean(losses))}
        if ep%C.eval_every==0 or ep==C.epochs:
            eval_models=models if ema is None else [ema]
            probs=predict_probs(eval_models,xe,be,method,C)
            yp=probs.argmax(1).numpy()
            rec.update(test_acc=float(accuracy_score(ytest,yp)*100),test_macro_f1=float(f1_score(ytest,yp,average='macro',zero_division=0)*100))
            print(f'{method} fold{fold} ep{ep}: acc={rec["test_acc"]:.2f}% F1={rec["test_macro_f1"]:.2f}%')
        hist.append(rec)
        if ep%C.ckpt_every==0 or ep==C.epochs:
            durable_write(RESULTS_DIR/f'{method}_fold{fold}_history.csv',pd.DataFrame(hist).to_csv(index=False))
            # Save intermediate checkpoint for resume; deleted after final JSON.
            state={'epoch':ep,'models':[m.state_dict() for m in models],
                   'opts':[o.state_dict() for o in opts],'sched':[s.state_dict() for s in sch],
                   'scalers':[s.state_dict() for s in scalers],
                   'ema':ema.state_dict() if ema is not None else None,'history':hist,
                   'python_rng':random.getstate(),'numpy_rng':np.random.get_state(),
                   'torch_rng':torch.get_rng_state(),'cuda_rng':torch.cuda.get_rng_state_all()}
            durable_torch_save(state,ckpt)
    final_models=models if ema is None else [ema]
    probs=predict_probs(final_models,xe,be,method,C)
    yp=probs.argmax(1).numpy()
    cm=confusion_matrix(ytest,yp,labels=list(range(50))).tolist()
    npar=sum(n_params(m) for m in models)
    result={'method':method,'model':'resnet_tiny','fold':fold,'epoch':C.epochs,
            'final_acc':float(accuracy_score(ytest,yp)*100),
            'final_macro_f1':float(f1_score(ytest,yp,average='macro',zero_division=0)*100),
            'final_weighted_f1':float(f1_score(ytest,yp,average='weighted',zero_division=0)*100),
            'parameters':int(npar),'seconds':time.time()-start_time,
            'y_true':ytest.tolist(),'y_pred':yp.tolist(),'confusion_matrix':cm,
            'history':hist,'config':vars(C),'description':'P1 ResNet-tiny + EfficientNet-B0, two log-mel views, fixed equal-probability ensemble' if ema is None else 'P2 CNN Transformer EMA and fixed 3-view TTA'}
    if SAVE_WEIGHTS:
        durable_torch_save({'models':[m.state_dict() for m in models], 'ema':ema.state_dict() if ema is not None else None},RESULTS_DIR/f'{method}_fold{fold}.pt')
    save_json(out,result)     # JSON is completion marker; write last
    if ckpt.exists():ckpt.unlink()
    print('SAVED',out,'ACC',result['final_acc'])
    del models,opts,sch,scalers,xa,xe,xb,be
    if torch.cuda.is_available():torch.cuda.empty_cache()
    return result


## 4. Run P1 and P2 — 5 folds × 500 epochs each

In [12]:
# The first method uses TWO independently trained ResNet-tiny models per fold.
# Disk: cache in esc50_repro/; results in /kaggle/working/outputs_advanced/results.
for method in RUN_METHODS:
    print('===',method,'===')
    a=build_dataset(C)
    b=build_dataset(C_ALT) if method=='P1_ensemble' else None
    for fold in RUN_FOLDS:
        run_advanced_fold(method,fold,a,b)
    del a,b
    torch.cuda.empty_cache()
print('All requested folds complete.')

=== P1_ensemble ===
loading cache esc50_spec_ca8ea0d9.pt


loading cache esc50_spec_706b1363.pt
Resuming P1_ensemble 1 epoch 101


P1_ensemble fold1:  12%|█▏        | 49/400 [01:43<13:24,  2.29s/it]

P1_ensemble fold1 ep150: acc=89.50% F1=89.27%


P1_ensemble fold1:  25%|██▍       | 99/400 [03:41<11:45,  2.34s/it]

P1_ensemble fold1 ep200: acc=91.75% F1=91.57%


... (208 outputs left out)

P2_hybrid_ema_tta fold5: 100%|█████████▉| 499/500 [05:23<00:00,  1.57it/s]

P2_hybrid_ema_tta fold5 ep500: acc=85.25% F1=84.43%


P2_hybrid_ema_tta fold5: 100%|██████████| 500/500 [05:24<00:00,  1.54it/s]


SAVED /root/acdnet/outputs_advanced/results/P2_hybrid_ema_tta_fold5.json ACC 85.25
All requested folds complete.


## 5. Save summary, portable ZIP and verify files for Kaggle Output

In [13]:
rows=[]
for p in sorted(RESULTS_DIR.glob('P*_fold*.json')):
    r=json.loads(p.read_text());rows.append({'method':r['method'],'fold':r['fold'],'accuracy':r['final_acc'],
                                             'macro_f1':r['final_macro_f1'],'parameters':r['parameters'],'seconds':r['seconds']})
df=pd.DataFrame(rows)
print(df.to_string(index=False) if not df.empty else 'No fold results yet')
if not df.empty:
    durable_write(OUT_DIR/'summary.csv',df.to_csv(index=False))
    print(df.groupby('method')[['accuracy','macro_f1']].agg(['mean','std']))
# Portable archive includes JSON and CSV only; checkpoints/feature caches are excluded.
archive=shutil.make_archive(str(PROJECT_DIR/'esc50_advanced_results'),'zip',root_dir=OUT_DIR,base_dir='results')
print('ZIP:',archive,Path(archive).stat().st_size,'bytes')
print('JSON files:',len(list(RESULTS_DIR.glob('P*_fold*.json'))))
print('Expected for full run: 10 JSON files.')
print('Kaggle saved-version output: outputs_advanced/results and esc50_advanced_results.zip')


           method  fold  accuracy  macro_f1  parameters     seconds
      P1_ensemble     1     92.00 91.691788     7112576  939.496955
      P1_ensemble     2     87.75 87.287995     7112576 1205.541864
      P1_ensemble     3     89.00 88.935636     7112576 1214.136894
      P1_ensemble     4     91.75 91.475655     7112576 1218.061666
      P1_ensemble     5     84.75 83.469460     7112576 1220.959828
P2_hybrid_ema_tta     1     86.50 85.719379     2685778  327.054697
P2_hybrid_ema_tta     2     86.00 85.330490     2685778  325.297854
P2_hybrid_ema_tta     3     88.75 88.563461     2685778  327.007537
P2_hybrid_ema_tta     4     88.50 88.226210     2685778  334.173974
P2_hybrid_ema_tta     5     85.25 84.430134     2685778  324.526891
                  accuracy             macro_f1          
                      mean       std       mean       std
method                                                   
P1_ensemble          89.05  3.007283  88.572107  3.390468
P2_hybrid_ema_tta   

ZIP: /root/acdnet/esc50_advanced_results.zip 234046843 bytes
JSON files: 10
Expected for full run: 10 JSON files.
Kaggle saved-version output: outputs_advanced/results and esc50_advanced_results.zip
